# Phân Tích Không Gian Embedding & Cross-Modal Alignment

**Mục tiêu:** Đi sâu vào cấu trúc hình học của không gian embedding đa phương thức,
phân tích độ lệch miền (domain gap), hiện tượng hubness, sự suy giảm ngữ nghĩa
theo độ phức tạp query, và temporal decay của visual concepts.

**Embedding models:**
- Visual: `timm/PE-Core-bigG-14-448` → 1280 chiều (L2 normalized)
- Transcript: `intfloat/multilingual-e5-small` → 384 chiều (L2 normalized)

**Ý nghĩa thực tiễn:** Mỗi phân tích ở đây ánh xạ trực tiếp đến một quyết định
kiến trúc trong backend fusion strategy (RRF weights, hubness penalty,
temporal decay multipliers, query complexity routing).

## 0. Cấu Hình & Mock Data Generation

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from scipy.spatial.distance import cdist
from scipy.stats import pearsonr
from sklearn.decomposition import PCA, TruncatedSVD
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler, normalize
from sklearn.metrics.pairwise import cosine_similarity, cosine_distances
from sklearn.feature_extraction.text import CountVectorizer
import random
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.05)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['figure.figsize'] = (13, 6)
np.random.seed(2026)
random.seed(2026)

# ─── Constants ────────────────────────────────────────────────────────────
N_FRAMES = 200
N_TRANSCRIPTS = 80
N_VIDEOS = 4
VISUAL_DIM = 1280
TRANSCRIPT_DIM = 384
N_TOPICS = 14

TOPICS = [
    'Ẩm thực', 'Công nghệ', 'Du lịch', 'Thể thao',
    'Giáo dục', 'Kinh tế', 'Sức khỏe', 'Giải trí',
    'Thời sự', 'Văn hóa', 'Đời sống', 'Môi trường',
    'Giao thông', 'Pháp luật'
]

VIDEO_IDS = ['L01_V001', 'L01_V002', 'L01_V003', 'L01_V004']
FPS = 25.0

print(f'Cấu hình: {N_FRAMES} frames × {N_TRANSCRIPTS} transcripts × {N_VIDEOS} videos × {N_TOPICS} topics')

In [ ]:
# ─── Mock Embedding Generator với cấu trúc topic-aware ────────────────────
#
# Thay vì sinh embedding hoàn toàn ngẫu nhiên, ta tạo các "topic centroid" 
# để mô phỏng cấu trúc ngữ nghĩa thực tế: cùng topic → embedding gần nhau.
# Đồng thời thêm "domain shift" — visual centroid và text centroid cùng topic 
# lệch nhau một khoảng để phản ánh cross-modal gap.

def make_topic_centroids(n_topics, dim, seed=42):
    """Tạo N centroid trực giao trong không gian embedding."""
    rng = np.random.RandomState(seed)
    centroids = rng.randn(n_topics, dim).astype(np.float32)
    centroids = centroids / np.linalg.norm(centroids, axis=1, keepdims=True)
    return centroids

def make_embeddings(n_samples, dim, topics, topic_centroids, within_topic_std=0.15, seed=42):
    """Sinh embedding quanh topic centroid với noise."""
    rng = np.random.RandomState(seed)
    embeddings = np.zeros((n_samples, dim), dtype=np.float32)
    assigned_topics = []
    for i in range(n_samples):
        t = topics[i]
        center = topic_centroids[t]
        noise = rng.randn(dim).astype(np.float32) * within_topic_std
        emb = center + noise
        emb = emb / (np.linalg.norm(emb) + 1e-8)
        embeddings[i] = emb
        assigned_topics.append(t)
    return embeddings, assigned_topics

# Visual centroid (1280d) và text centroid (384d) — khác không gian nhưng cùng chỉ số topic
vis_centroids_1280 = make_topic_centroids(N_TOPICS, VISUAL_DIM, seed=1)
txt_centroids_384 = make_topic_centroids(N_TOPICS, TRANSCRIPT_DIM, seed=1)

# ─── Sinh frames ──────────────────────────────────────────────────────────

frame_topics = [random.randrange(N_TOPICS) for _ in range(N_FRAMES)]
vis_embeddings, _ = make_embeddings(N_FRAMES, VISUAL_DIM, frame_topics, vis_centroids_1280,
                                     within_topic_std=0.12, seed=42)

frames = []
for i in range(N_FRAMES):
    vid = VIDEO_IDS[i % N_VIDEOS]
    frame_num = random.randint(0, 3000)
    ts_ms = int(frame_num / FPS * 1000)
    frames.append({
        'frame_id': f'{vid}_{frame_num:06d}',
        'video_id': vid,
        'frame_number': frame_num,
        'timestamp_ms': ts_ms,
        'topic_idx': frame_topics[i],
        'topic': TOPICS[frame_topics[i]],
        'embedding': vis_embeddings[i],
    })

# ─── Sinh transcripts ─────────────────────────────────────────────────────

transcript_topics = [random.randrange(N_TOPICS) for _ in range(N_TRANSCRIPTS)]
txt_embeddings, _ = make_embeddings(N_TRANSCRIPTS, TRANSCRIPT_DIM, transcript_topics,
                                     txt_centroids_384, within_topic_std=0.10, seed=2026)

transcripts = []
for i in range(N_TRANSCRIPTS):
    vid = VIDEO_IDS[i % N_VIDEOS]
    seg_len = random.randint(3000, 20000)
    start_ms = random.randint(0, 120000 - seg_len)
    transcripts.append({
        'video_id': vid,
        'start_time_ms': start_ms,
        'end_time_ms': start_ms + seg_len,
        'topic_idx': transcript_topics[i],
        'topic': TOPICS[transcript_topics[i]],
        'embedding': txt_embeddings[i],
    })

df_frames = pd.DataFrame([{k: v for k, v in f.items() if k != 'embedding'} for f in frames])
df_transcripts = pd.DataFrame([{k: v for k, v in t.items() if k != 'embedding'} for t in transcripts])

print(f'Frames:    {len(frames)} | Topics: {df_frames["topic"].nunique()}')
print(f'Transcripts: {len(transcripts)} | Topics: {df_transcripts["topic"].nunique()}')

---
## 1. Embedding Space Topology & Domain Gap (t-SNE)

**Mục tiêu:** Trực quan hóa khoảng cách giữa visual embedding (1280d) và transcript
embedding (384d) trong một không gian chung sau khi giảm chiều.

**Phương pháp:**
1. Dùng `TruncatedSVD` giảm cả hai embedding về 64 chiều chung (shared latent space).
2. Nối hai tập đã giảm → chạy t-SNE xuống 2D.
3. Color-code theo 14 chủ đề, dùng marker khác nhau cho visual (●) và text (▲).

**Kỳ vọng:** Cùng chủ đề → visual và text tạo thành 2 cụm gần nhau nhưng không
trùng lặp hoàn toàn, phản ánh "domain gap" cố hữu giữa ảnh và văn bản.

In [ ]:
from sklearn.decomposition import TruncatedSVD

SHARED_DIM = 64  # Không gian latent chung

# ─── Giảm chiều riêng từng modality về shared_dim ─────────────────────────

vis_matrix = np.stack([f['embedding'] for f in frames])
txt_matrix = np.stack([t['embedding'] for t in transcripts])

svd_vis = TruncatedSVD(n_components=SHARED_DIM, random_state=42)
svd_txt = TruncatedSVD(n_components=SHARED_DIM, random_state=42)

vis_reduced = svd_vis.fit_transform(vis_matrix)
txt_reduced = svd_txt.fit_transform(txt_matrix)

print(f'Visual:  {vis_matrix.shape} → {vis_reduced.shape}  (explained var: {svd_vis.explained_variance_ratio_.sum():.3f})')
print(f'Text:    {txt_matrix.shape} → {txt_reduced.shape}  (explained var: {svd_txt.explained_variance_ratio_.sum():.3f})')

# ─── Chuẩn hóa để tránh modality scale bias trước t-SNE ───────────────────

scaler = StandardScaler()
vis_norm = scaler.fit_transform(vis_reduced)
txt_norm = scaler.fit_transform(txt_reduced)

# ─── t-SNE trên tập hợp nhất ──────────────────────────────────────────────

combined = np.vstack([vis_norm, txt_norm])
modality_labels = np.array(['Visual'] * N_FRAMES + ['Transcript'] * N_TRANSCRIPTS)
all_topics = [f['topic'] for f in frames] + [t['topic'] for t in transcripts]

import sklearn
tsne_kwargs = {'n_components': 2, 'perplexity': 30, 'random_state': 42, 'metric': 'cosine'}
if sklearn.__version__ >= '1.7.0':
    tsne_kwargs['max_iter'] = 1000
else:
    tsne_kwargs['n_iter'] = 1000
tsne = TSNE(**tsne_kwargs)
coords_2d = tsne.fit_transform(combined)

df_tsne = pd.DataFrame({
    'x': coords_2d[:, 0],
    'y': coords_2d[:, 1],
    'modality': modality_labels,
    'topic': all_topics,
})
print('t-SNE hoàn tất.')

In [ ]:
# ─── Trực quan hóa Domain Gap ─────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

# Panel 1: Color theo modality
for mod, color, marker, alpha in [('Visual', '#e74c3c', 'o', 0.35),
                                   ('Transcript', '#3498db', '^', 0.55)]:
    mask = df_tsne['modality'] == mod
    axes[0].scatter(df_tsne.loc[mask, 'x'], df_tsne.loc[mask, 'y'],
                    c=color, marker=marker, alpha=alpha, s=35, edgecolors='none',
                    label=f'{mod} ({mask.sum()})')

axes[0].set_title('Không Gian Embedding Sau t-SNE\nColor = Modality',
                  fontweight='bold', fontsize=13)
axes[0].legend(markerscale=1.5, fontsize=10)
axes[0].set_xlabel('t-SNE 1')
axes[0].set_ylabel('t-SNE 2')

# Panel 2: Color theo topic (14 màu)
import matplotlib.cm as cm
topic_colors = cm.tab20(np.linspace(0, 1, N_TOPICS))
for t_idx, topic_name in enumerate(TOPICS):
    mask = df_tsne['topic'] == topic_name
    axes[1].scatter(df_tsne.loc[mask, 'x'], df_tsne.loc[mask, 'y'],
                    color=topic_colors[t_idx], s=20, alpha=0.6,
                    edgecolors='none', label=topic_name)

axes[1].set_title('Không Gian Embedding Sau t-SNE\nColor = 14 Chủ Đề',
                  fontweight='bold', fontsize=13)
axes[1].set_xlabel('t-SNE 1')
axes[1].set_ylabel('t-SNE 2')
axes[1].legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=7.5,
               title='Chủ đề', title_fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ─── Đo lường định lượng Domain Gap ───────────────────────────────────────
#
# Với mỗi topic, tính:
#   - Intra-modality distance: khoảng cách trung bình giữa các visual (hoặc text) cùng topic
#   - Cross-modality distance: khoảng cách trung bình giữa visual và text cùng topic
#   - Domain Gap Ratio = cross_distance / intra_distance
#     > 1 → cross-modal gap lớn hơn intra-class variance

def compute_domain_gap(vis_embs, txt_embs, vis_topics, txt_topics, topics):
    """Tính Domain Gap ratio cho từng chủ đề."""
    results = []
    for t_idx, t_name in enumerate(topics):
        v_mask = np.array([t == t_idx for t in vis_topics])
        t_mask = np.array([t == t_idx for t in txt_topics])

        if v_mask.sum() < 2 or t_mask.sum() < 2:
            continue

        v_emb = np.stack([vis_embs[i] for i in range(len(vis_embs)) if v_mask[i]])
        t_emb = np.stack([txt_embs[i] for i in range(len(txt_embs)) if t_mask[i]])

        # Intra-visual distance
        intra_v = np.mean(cosine_distances(v_emb)[np.triu_indices(len(v_emb), k=1)])
        # Intra-text distance
        intra_t = np.mean(cosine_distances(t_emb)[np.triu_indices(len(t_emb), k=1)])
        # Cross-modal distance (visual ↔ text, dùng PCA-reduced)
        cross_dists = cosine_distances(v_emb, t_emb)
        cross = np.mean(cross_dists)

        gap_ratio = cross / ((intra_v + intra_t) / 2) if (intra_v + intra_t) > 0 else 1.0
        results.append({
            'topic': t_name,
            'intra_visual_dist': intra_v,
            'intra_text_dist': intra_t,
            'cross_modal_dist': cross,
            'domain_gap_ratio': gap_ratio,
            'n_vis': v_mask.sum(),
            'n_txt': t_mask.sum(),
        })
    return pd.DataFrame(results)

df_gap = compute_domain_gap(vis_reduced, txt_reduced, frame_topics, transcript_topics, TOPICS)

fig, ax = plt.subplots(figsize=(13, 6))
df_gap_sorted = df_gap.sort_values('domain_gap_ratio', ascending=True)

x = np.arange(len(df_gap_sorted))
width = 0.3
ax.barh(x - width/2, df_gap_sorted['intra_visual_dist'], width, label='Intra-Visual', color='#e74c3c', alpha=0.7)
ax.barh(x + width/2, df_gap_sorted['intra_text_dist'], width, label='Intra-Text', color='#3498db', alpha=0.7)
ax.scatter(df_gap_sorted['cross_modal_dist'], x, color='black', s=60, zorder=5, marker='D', label='Cross-Modal')

ax.set_yticks(x)
ax.set_yticklabels(df_gap_sorted['topic'])
ax.set_xlabel('Cosine Distance')
ax.set_title('Phân Tích Domain Gap Theo Chủ Đề\n(Intra-Modal vs Cross-Modal Cosine Distance)',
             fontweight='bold')
ax.legend(loc='lower right')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print('Domain Gap Ratio (cao = lệch nhiều giữa visual và text):')
display(df_gap_sorted[['topic', 'domain_gap_ratio', 'intra_visual_dist', 'intra_text_dist', 'cross_modal_dist']])

> **Insight kiến trúc:** Chủ đề có `domain_gap_ratio` cao (visual và text xa nhau)
> → nên tăng trọng số cho text modality trong fusion strategy, vì visual matching kém tin cậy.
> Ngược lại, chủ đề có ratio thấp → visual search đã tốt, có thể giảm text weight.
> Đề xuất: Dùng `domain_gap_ratio` làm hệ số điều chỉnh per-topic trong `stable_fusion.py`.

---
## 2. Hubness Problem & Modality Bias Detection

**Bối cảnh:** Trong không gian high-dimensional, hiện tượng "hubness" xảy ra khi
một số điểm (frames) trở thành nearest neighbor của rất nhiều điểm khác — chúng
là "hubs". Trong retrieval, hubs gây giảm precision vì các frame chung chung
luôn xuất hiện trong top-K, bất kể query.

**Phương pháp:**
1. Tính ma trận cosine distance toàn bộ giữa transcripts (query) và frames (gallery).
2. Với mỗi transcript, lấy Top-K frame gần nhất.
3. Đếm tần suất mỗi frame xuất hiện trong Top-K.
4. Vẽ phân phối hubness. Frame xuất hiện > µ + 2σ lần → nghi ngờ là hub.

In [ ]:
K_HUB = 10  # Top-K để phân tích hubness

# ─── Cosine distance matrix: transcripts × frames ────────────────────────
# Dùng reduced embedding (64d shared space) để tính cross-modal distance

cross_dist_matrix = cosine_distances(txt_reduced, vis_reduced)  # shape (N_TXT, N_VIS)

# ─── Với mỗi transcript, lấy index của K frame gần nhất ──────────────────

hub_counter = Counter()
for i in range(N_TRANSCRIPTS):
    nearest_frame_indices = np.argpartition(cross_dist_matrix[i], K_HUB)[:K_HUB]
    for idx in nearest_frame_indices:
        hub_counter[idx] += 1

# ─── DataFrame hubness ────────────────────────────────────────────────────

hubness = np.array([hub_counter.get(i, 0) for i in range(N_FRAMES)])
df_hubness = pd.DataFrame({
    'frame_id': [f['frame_id'] for f in frames],
    'topic': [f['topic'] for f in frames],
    'hubness': hubness,
})

mean_h = hubness.mean()
std_h = hubness.std()
hub_threshold = mean_h + 2 * std_h
n_hubs = (hubness > hub_threshold).sum()

print(f'Phân tích Hubness (Top-{K_HUB}):')
print(f'  Trung bình: {mean_h:.2f} lần / frame')
print(f'  Ngưỡng hub (μ + 2σ): {hub_threshold:.2f}')
print(f'  Số frame nghi ngờ là hub: {n_hubs} / {N_FRAMES} ({n_hubs/N_FRAMES*100:.1f}%)')
print(f'  Hub lớn nhất: {hubness.max()} lần')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ─── Histogram hubness ────────────────────────────────────────────────────

axes[0].hist(hubness, bins=40, color='#8e44ad', edgecolor='white', alpha=0.8)
axes[0].axvline(hub_threshold, color='red', linestyle='--', linewidth=2.5,
                label=f'Hub threshold = {hub_threshold:.1f} (μ + 2σ)')
axes[0].axvline(mean_h, color='gray', linestyle=':', linewidth=2, label=f'Mean = {mean_h:.1f}')
axes[0].set_title(f'Phân Phối Hubness Của Frame\n(Top-{K_HUB}, {N_FRAMES} frames)',
                  fontweight='bold')
axes[0].set_xlabel(f'Số lần xuất hiện trong Top-{K_HUB} của transcripts')
axes[0].set_ylabel('Số frame')
axes[0].legend()

# ─── Phân bố hubness theo chủ đề ──────────────────────────────────────────

topic_hubness = df_hubness.groupby('topic')['hubness'].agg(['mean', 'std', 'max', 'count'])
topic_hubness = topic_hubness.sort_values('mean', ascending=False)

bars = axes[1].barh(topic_hubness.index, topic_hubness['mean'],
                    xerr=topic_hubness['std'], capsize=3,
                    color=plt.cm.viridis(np.linspace(0.2, 0.9, len(topic_hubness))),
                    edgecolor='white')
axes[1].set_title('Hubness Trung Bình Theo Chủ Đề\n(Chủ đề có hubness cao = frame dễ dominate retrieval)',
                  fontweight='bold')
axes[1].set_xlabel(f'Số lần trung bình trong Top-{K_HUB}')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# ─── Top hub frames ───────────────────────────────────────────────────────

print('\nTop 10 frame nghi ngờ là hub:')
display(df_hubness.nlargest(10, 'hubness')[['frame_id', 'topic', 'hubness']])

# ─── Kiểm tra: có phải hub tập trung vào 1-2 chủ đề không? ───────────────

hub_frame_topics = df_hubness[df_hubness['hubness'] > hub_threshold]['topic']
if len(hub_frame_topics) > 0:
    hub_topic_dist = hub_frame_topics.value_counts()
    print(f'\nPhân bố chủ đề của hub frames:')
    for topic, cnt in hub_topic_dist.items():
        print(f'  {topic}: {cnt}')
else:
    print('\nKhông phát hiện hub frame (phân phối đồng đều).')

> **Insight kiến trúc:** Nếu hubness tập trung vào một số frame chung chung
> (vd: "frame đen", "frame mờ", "frame nền đơn giản"), cần áp dụng **hubness penalty**
> trong backend: giảm confidence của frame có hubness cao bằng cách nhân với
> `1 / (1 + log(1 + hubness))`. Ngoài ra, hubness theo chủ đề gợi ý rằng một số
> chủ đề có embedding quality kém hơn → cần re-embed bằng model mạnh hơn
> hoặc data augmentation.

---
## 3. Query Complexity & Semantic Degradation

**Bối cảnh:** Không phải mọi query đều có độ khó như nhau. Query đơn giản
("xe máy") dễ matching hơn query phức tạp ("xe máy màu đỏ ở bên trái tòa nhà").

**Phân loại query:**
| Level | Mô tả | Ví dụ |
|---|---|---|
| **L1 — Object-only** | Chỉ đề cập đối tượng | "xe máy", "bác sĩ" |
| **L2 — Action/Attributive** | Đối tượng + hành động/thuộc tính | "người đang chạy xe máy màu đỏ" |
| **L3 — Spatial/Reasoning** | Có quan hệ không gian hoặc suy luận | "xe máy ở bên trái tòa nhà, phía sau có cây" |

**Kỳ vọng:** Score giảm dần L1 → L2 → L3, và phương sai tăng dần.

In [ ]:
# ─── Mô phỏng query với 3 mức độ phức tạp ────────────────────────────────

N_QUERIES_PER_LEVEL = 60

def simulate_query_score(complexity_level, n_queries, seed=42):
    """
    Mô phỏng similarity score dựa trên độ phức tạp.
    L1 (Object-only)     → mean cao, std thấp.
    L2 (Action/Attrib)   → mean trung bình, std trung bình.
    L3 (Spatial/Reason)  → mean thấp, std cao.
    """
    rng = np.random.RandomState(seed)
    if complexity_level == 'L1 — Object-only':
        scores = rng.beta(a=6, b=2, size=n_queries)  # Lệch phải, tập trung cao
    elif complexity_level == 'L2 — Action/Attributive':
        scores = rng.beta(a=4, b=3, size=n_queries)  # Phân tán hơn
    else:  # L3 — Spatial/Reasoning
        scores = rng.beta(a=2.5, b=3.5, size=n_queries)  # Lệch trái, phân tán rộng
    return np.clip(scores, 0.0, 1.0)

df_query = pd.DataFrame({
    'complexity': ['L1 — Object-only'] * N_QUERIES_PER_LEVEL +
                  ['L2 — Action/Attributive'] * N_QUERIES_PER_LEVEL +
                  ['L3 — Spatial/Reasoning'] * N_QUERIES_PER_LEVEL,
})
df_query['similarity_score'] = np.concatenate([
    simulate_query_score('L1 — Object-only', N_QUERIES_PER_LEVEL, seed=1),
    simulate_query_score('L2 — Action/Attributive', N_QUERIES_PER_LEVEL, seed=2),
    simulate_query_score('L3 — Spatial/Reasoning', N_QUERIES_PER_LEVEL, seed=3),
])

print('Thống kê similarity score theo độ phức tạp query:')
display(df_query.groupby('complexity')['similarity_score'].describe().round(4))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# ─── Box plot ─────────────────────────────────────────────────────────────

order = ['L1 — Object-only', 'L2 — Action/Attributive', 'L3 — Spatial/Reasoning']
palette = {'L1 — Object-only': '#2ecc71',
           'L2 — Action/Attributive': '#f39c12',
           'L3 — Spatial/Reasoning': '#e74c3c'}

bp = sns.boxplot(data=df_query, x='complexity', y='similarity_score',
                 order=order, palette=palette, width=0.5, linewidth=1.5, ax=axes[0])
bp = sns.stripplot(data=df_query, x='complexity', y='similarity_score',
                   order=order, color='black', size=3, alpha=0.3, jitter=True, ax=axes[0])
axes[0].set_title('Semantic Degradation Theo Độ Phức Tạp Query', fontweight='bold', fontsize=13)
axes[0].set_xlabel('')
axes[0].set_ylabel('Similarity Score')

# ─── Violin plot ──────────────────────────────────────────────────────────

sns.violinplot(data=df_query, x='complexity', y='similarity_score',
               order=order, palette=palette, inner='quartile', ax=axes[1])
axes[1].set_title('Phân Phối Score Theo Query Complexity\n(Violin — Median + Quartiles)',
                  fontweight='bold', fontsize=13)
axes[1].set_xlabel('')
axes[1].set_ylabel('Similarity Score')

# ─── Bar chart: Mean ± Std ────────────────────────────────────────────────

summary = df_query.groupby('complexity')['similarity_score'].agg(['mean', 'std'])
summary = summary.loc[order]
bars = axes[2].bar(summary.index, summary['mean'],
                   yerr=summary['std'], capsize=8, color=[palette[l] for l in order],
                   edgecolor='white', width=0.5)
for bar, mean_val in zip(bars, summary['mean']):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{mean_val:.3f}', ha='center', fontweight='bold', fontsize=11)
axes[2].set_title('Mean Similarity ± Std', fontweight='bold', fontsize=13)
axes[2].set_ylabel('Similarity Score')
axes[2].set_ylim(0, 1)

plt.tight_layout()
plt.show()

# ─── Độ suy giảm ─────────────────────────────────────────────────────────

mean_l1 = summary.loc['L1 — Object-only', 'mean']
mean_l2 = summary.loc['L2 — Action/Attributive', 'mean']
mean_l3 = summary.loc['L3 — Spatial/Reasoning', 'mean']
print(f'Độ suy giảm semantic:')
print(f'  L1 → L2: {(mean_l1 - mean_l2) / mean_l1 * 100:.1f}% giảm')
print(f'  L2 → L3: {(mean_l2 - mean_l3) / mean_l2 * 100:.1f}% giảm')
print(f'  L1 → L3: {(mean_l1 - mean_l3) / mean_l1 * 100:.1f}% giảm tổng')

> **Insight kiến trúc:** Query càng phức tạp → visual-only matching càng kém.
> Đề xuất triển khai **query complexity classifier** ở `pre_process()`:
> - L1 → visual weight cao (0.8), text weight thấp (0.2)
> - L2 → cân bằng (0.5 visual + 0.3 transcript + 0.2 OCR)
> - L3 → text-heavy (0.3 visual + 0.4 transcript + 0.3 OCR) + temporal context
>
> Ngoài ra, L3 có std cao → cần confidence calibration hoặc fallback strategy.

---
## 4. OCR vs Transcript — Giao Thoa & Bổ Sung Thông Tin

**Bối cảnh:** OCR (text trên frame) và Transcript (lời nói) là hai nguồn text
khác nhau. Khi chúng overlap cao → thông tin dư thừa (redundant). Khi overlap
thấp → hai nguồn bổ sung cho nhau (complementary), tăng coverage.

**Phương pháp:**
1. Sinh mock OCR text cho mỗi frame.
2. Temporal-map frame với transcript.
3. Tính Jaccard similarity và word overlap ratio giữa OCR và transcript.
4. Phân tích tương quan với visual-semantic score.

In [ ]:
# ─── Sinh OCR text cho frames ─────────────────────────────────────────────

OCR_VOCAB_BY_TOPIC = {
    'Ẩm thực':   ['phở', 'bún', 'cơm', 'nước', 'canh', 'thịt', 'cá', 'rau', 'quán', 'bếp'],
    'Công nghệ':  ['AI', 'chip', 'code', 'data', 'server', 'cloud', 'app', 'web', 'API', 'bot'],
    'Thể thao':   ['bàn thắng', 'penalty', 'vàng', 'bạc', 'đồng', 'vô địch', 'tỷ số', 'hiệp', 'sân', 'bóng'],
    'Giáo dục':   ['bài tập', 'điểm', 'thi', 'lớp', 'trường', 'sách', 'bảng', 'viết', 'học', 'toán'],
    'Thời sự':    ['tin', 'báo', 'họp', 'quốc hội', 'thủ tướng', 'phát biểu', 'nghị', 'án', 'dự', 'luật'],
    'Kinh tế':    ['giá', 'USD', 'VNĐ', 'chứng khoán', 'lãi', 'vốn', 'thị trường', 'bán', 'mua', 'hàng'],
    'Sức khỏe':   ['thuốc', 'bệnh', 'viện', 'máu', 'tim', 'phổi', 'vaccine', 'khám', 'đơn', 'BS'],
    'Giải trí':   ['phim', 'nhạc', 'show', 'vé', 'sân khấu', 'diễn', 'hát', 'kịch', 'rap', 'beat'],
    'Du lịch':    ['tour', 'vé', 'bay', 'khách sạn', 'resort', 'check-in', 'view', 'biển', 'núi', 'đảo'],
    'Văn hóa':    ['lễ', 'hội', 'chùa', 'đền', 'tranh', 'tượng', 'múa', 'hát', 'trống', 'cồng'],
    'Đời sống':   ['chợ', 'siêu thị', 'giá', 'mua', 'sắm', 'nhà', 'cửa', 'xe', 'điện', 'nước'],
    'Môi trường': ['xanh', 'sạch', 'rác', 'nước', 'không khí', 'cây', 'rừng', 'biển', 'CO2', 'khí'],
    'Giao thông': ['km', 'giờ', 'đường', 'cầu', 'phà', 'xe', 'bus', 'tàu', 'ga', 'bến'],
    'Pháp luật':  ['điều', 'khoản', 'luật', 'tòa', 'án', 'xử', 'phạt', 'tù', 'công an', 'bằng'],
}

def generate_ocr_text(topic, n_words=None):
    """Sinh OCR text ngẫu nhiên dựa trên vocab của chủ đề."""
    vocab = OCR_VOCAB_BY_TOPIC.get(topic, ['text'])
    if n_words is None:
        n_words = random.randint(1, 5)
    return ' '.join(random.sample(vocab, min(n_words, len(vocab))))

# ─── Temporal-map frame → transcript rồi gán OCR ──────────────────────────

def temporal_map_with_ocr(frames, transcripts, tolerance_ms=3000):
    """Map frame → transcript + sinh OCR."""
    records = []
    for f in frames:
        vid = f['video_id']
        ts = f['timestamp_ms']
        matched = False
        for t in transcripts:
            if t['video_id'] != vid:
                continue
            if t['start_time_ms'] - tolerance_ms <= ts <= t['end_time_ms'] + tolerance_ms:
                records.append({
                    'frame_id': f['frame_id'],
                    'topic': f['topic'],
                    'ocr_text': generate_ocr_text(f['topic']),
                    'transcript_text': generate_ocr_text(f['topic'], n_words=random.randint(8, 25)),
                    'visual_score': np.clip(random.betavariate(3, 2.5), 0.1, 0.95),
                })
                matched = True
                break
        if not matched:
            records.append({
                'frame_id': f['frame_id'],
                'topic': f['topic'],
                'ocr_text': generate_ocr_text(f['topic']),
                'transcript_text': '',
                'visual_score': np.clip(random.betavariate(3, 2.5), 0.1, 0.95),
            })
    return pd.DataFrame(records)

df_ocr = temporal_map_with_ocr(frames, transcripts)

# ─── Tính Jaccard similarity và Word Overlap ──────────────────────────────

def jaccard_similarity(text_a, text_b):
    """Jaccard similarity giữa hai tập từ."""
    set_a = set(text_a.lower().split())
    set_b = set(text_b.lower().split())
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def word_overlap_ratio(text_a, text_b):
    """Tỷ lệ từ trong OCR xuất hiện trong transcript."""
    set_a = set(text_a.lower().split())
    set_b = set(text_b.lower().split())
    if not set_a:
        return 0.0
    return len(set_a & set_b) / len(set_a)

df_ocr['jaccard_ocr_transcript'] = df_ocr.apply(
    lambda r: jaccard_similarity(r['ocr_text'], r['transcript_text']), axis=1)
df_ocr['word_overlap_ratio'] = df_ocr.apply(
    lambda r: word_overlap_ratio(r['ocr_text'], r['transcript_text']), axis=1)

print(f'Jaccard trung bình: {df_ocr["jaccard_ocr_transcript"].mean():.4f}')
print(f'Word overlap trung bình: {df_ocr["word_overlap_ratio"].mean():.4f}')
print(f'Số cặp overlap = 0 (bổ sung hoàn toàn): {(df_ocr["word_overlap_ratio"] == 0).sum()} / {len(df_ocr)}')
print(f'Số cặp overlap > 0.5 (dư thừa): {(df_ocr["word_overlap_ratio"] > 0.5).sum()} / {len(df_ocr)}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

# ─── Scatter: Jaccard OCR-Transcript vs Visual Score ────────────────────

sc = axes[0].scatter(df_ocr['jaccard_ocr_transcript'], df_ocr['visual_score'],
                     c=df_ocr['word_overlap_ratio'], cmap='RdYlGn',
                     alpha=0.6, s=60, edgecolors='white', linewidth=0.3)
axes[0].set_xlabel('Jaccard Similarity (OCR ↔ Transcript)')
axes[0].set_ylabel('Visual-Semantic Score')
axes[0].set_title('OCR–Transcript Overlap vs Visual Score', fontweight='bold')
cbar = plt.colorbar(sc, ax=axes[0])
cbar.set_label('Word Overlap Ratio')

# ─── Histogram 2D ─────────────────────────────────────────────────────────

hist = axes[1].hist2d(df_ocr['jaccard_ocr_transcript'], df_ocr['visual_score'],
                      bins=20, cmap='YlOrRd')
axes[1].set_xlabel('Jaccard Similarity (OCR ↔ Transcript)')
axes[1].set_ylabel('Visual-Semantic Score')
axes[1].set_title('Mật Độ Phân Bố\nOverlap vs Score', fontweight='bold')
plt.colorbar(hist[3], ax=axes[1], label='Số frame')

# ─── Phân nhóm theo mức overlap ───────────────────────────────────────────

df_ocr['overlap_group'] = pd.cut(df_ocr['word_overlap_ratio'],
                                  bins=[-0.01, 0.0, 0.3, 1.01],
                                  labels=['Không overlap\n(bổ sung)',
                                          'Overlap thấp\n(mixed)',
                                          'Overlap cao\n(dư thừa)'])

order_grp = ['Không overlap\n(bổ sung)', 'Overlap thấp\n(mixed)', 'Overlap cao\n(dư thừa)']
sns.boxplot(data=df_ocr, x='overlap_group', y='visual_score',
            order=order_grp,
            palette={'Không overlap\n(bổ sung)': '#2ecc71',
                     'Overlap thấp\n(mixed)': '#f39c12',
                     'Overlap cao\n(dư thừa)': '#e74c3c'},
            ax=axes[2])
axes[2].set_title('Visual Score Theo Mức Overlap OCR–Transcript', fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('Visual-Semantic Score')

# ─── Thống kê ─────────────────────────────────────────────────────────────

overlap_stats = df_ocr.groupby('overlap_group', observed=False)['visual_score'].agg(['mean', 'std', 'count'])
print('Phân tích overlap OCR–Transcript:')
display(overlap_stats.round(4))

plt.tight_layout()
plt.show()

> **Insight kiến trúc:**
> - **Overlap cao → dư thừa:** OCR và transcript nói cùng một thứ. Nên giảm trọng số
>   của một trong hai để tránh double-counting trong fusion.
> - **Overlap thấp → bổ sung:** OCR cung cấp thông tin mà transcript không có. Nên
>   tăng trọng số tổng của text modality (vì coverage tăng).
> - **Đề xuất:** Thêm `overlap_penalty = 1 - overlap_ratio * 0.5` vào text fusion weight.
>   Khi overlap hoàn toàn, text weight giảm 50%; khi không overlap, text weight giữ nguyên.

---
## 5. Temporal Decay & Concept Persistence

**Bối cảnh:** Khi một keyword xuất hiện trong transcript tại thời điểm T,
nội dung hình ảnh thường không thay đổi ngay lập tức — concept tồn tại
một khoảng thời gian trước khi "phai nhạt". Hiểu được đường cong suy giảm
này giúp thiết kế temporal decay multiplier cho fusion strategy.

**Phương pháp:**
1. Chọn một event tại T (keyword xuất hiện trong transcript).
2. Đo visual similarity score của các frame tại T+0s, T+1s, ..., T+10s.
3. Fit đường cong exponential decay: `score(t) = A * exp(-λ * t) + baseline`.
4. Tính half-life: thời gian để score giảm còn một nửa.

In [ ]:
# ─── Mô phỏng temporal decay ──────────────────────────────────────────────

N_EVENTS = 50  # Số sự kiện transcript
MAX_TIME_S = 10  # Theo dõi trong 10 giây
TIME_STEP_S = 0.5  # Mỗi bước 0.5s

def simulate_decay_curve(a=0.85, lam=0.3, baseline=0.1, noise_std=0.04, seed=42):
    """
    Mô phỏng đường cong suy giảm semantic.
    score(t) = A * exp(-λ * t) + baseline + noise
    """
    rng = np.random.RandomState(seed)
    t = np.arange(0, MAX_TIME_S + TIME_STEP_S, TIME_STEP_S)
    scores = a * np.exp(-lam * t) + baseline
    scores += rng.randn(len(t)) * noise_std
    return np.clip(scores, 0.0, 1.0), t

# ─── Sinh nhiều decay curves với tham số hơi khác nhau ────────────────────

all_curves = []
all_times = None
for i in range(N_EVENTS):
    a_i = np.random.uniform(0.75, 0.95)
    lam_i = np.random.uniform(0.15, 0.50)
    baseline_i = np.random.uniform(0.05, 0.20)
    scores, times = simulate_decay_curve(a=a_i, lam=lam_i, baseline=baseline_i, seed=i)
    all_curves.append(scores)
    if all_times is None:
        all_times = times

curves_matrix = np.array(all_curves)  # (N_EVENTS, N_TIMESTEPS)
mean_curve = curves_matrix.mean(axis=0)
std_curve = curves_matrix.std(axis=0)

print(f'Mô phỏng {N_EVENTS} events × {len(times)} timesteps (0 – {MAX_TIME_S}s)')

In [ ]:
from scipy.optimize import curve_fit

# ─── Fit exponential decay lên đường trung bình ───────────────────────────

def exp_decay(t, A, lam, baseline):
    return A * np.exp(-lam * t) + baseline

popt, pcov = curve_fit(exp_decay, times, mean_curve, p0=[0.8, 0.3, 0.1], bounds=([0, 0, 0], [1, 2, 0.5]))
A_fit, lam_fit, baseline_fit = popt

# Tính half-life: thời gian để score giảm từ A → A/2 (tính từ baseline)
half_life = np.log(2) / lam_fit

print(f'Tham số decay curve (fit từ dữ liệu trung bình):')
print(f'  Amplitude (A):     {A_fit:.4f}')
print(f'  Decay rate (λ):    {lam_fit:.4f}')
print(f'  Baseline:          {baseline_fit:.4f}')
print(f'  Half-life (T½):    {half_life:.2f}s')
print(f'  → Sau {half_life:.1f}s, semantic score giảm còn một nửa so với đỉnh.')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6.5))

# ─── Đường decay trung bình + fit ─────────────────────────────────────────

ax = axes[0]
ax.fill_between(times, mean_curve - std_curve, mean_curve + std_curve,
                alpha=0.15, color='#3498db', label='± 1 Std')

# Vẽ một số đường mẫu
for i in range(min(N_EVENTS, 20)):
    ax.plot(times, curves_matrix[i], color='gray', alpha=0.15, linewidth=0.8)

ax.plot(times, mean_curve, color='#3498db', linewidth=3, label='Trung bình', zorder=5)
ax.plot(times, exp_decay(times, *popt), '--', color='#e74c3c', linewidth=3,
        label=f'Fit: A·e^(−λt)+b, T½={half_life:.1f}s', zorder=6)

# Đánh dấu half-life
ax.axvline(half_life, color='green', linestyle=':', linewidth=2, alpha=0.7)
ax.annotate(f'T½ = {half_life:.1f}s',
            xy=(half_life, exp_decay(half_life, *popt)),
            xytext=(half_life + 1.5, exp_decay(half_life, *popt) + 0.15),
            arrowprops=dict(arrowstyle='->', color='green'),
            fontsize=11, fontweight='bold', color='green')

ax.set_title('Temporal Decay Của Semantic Score\nKhi Keyword Xuất Hiện Trong Transcript Tại T=0',
             fontweight='bold', fontsize=13)
ax.set_xlabel('Thời gian sau keyword (giây)')
ax.set_ylabel('Similarity Score')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)

# ─── Phân phối half-life giữa các events ─────────────────────────────────

# Fit từng đường riêng để xem phân phối half-life
half_lives = []
for i in range(N_EVENTS):
    try:
        popt_i, _ = curve_fit(exp_decay, times, curves_matrix[i],
                              p0=[0.8, 0.3, 0.1], bounds=([0, 0.01, 0], [1, 1.5, 0.5]),
                              maxfev=5000)
        hl = np.log(2) / popt_i[1]
        if 0.5 < hl < 30:
            half_lives.append(hl)
    except:
        pass

axes[1].hist(half_lives, bins=20, color='#9b59b6', edgecolor='white', alpha=0.8)
axes[1].axvline(np.mean(half_lives), color='red', linestyle='--', linewidth=2.5,
                label=f'Mean = {np.mean(half_lives):.2f}s')
axes[1].set_title('Phân Phối Half-Life (T½) Giữa Các Events', fontweight='bold')
axes[1].set_xlabel('Half-life (giây)')
axes[1].set_ylabel('Số event')
axes[1].legend()

plt.tight_layout()
plt.show()

print(f'\nHalf-life distribution:')
if half_lives:
    print(f'  Mean:   {np.mean(half_lives):.2f}s')
    print(f'  Median: {np.median(half_lives):.2f}s')
    print(f'  Min:    {np.min(half_lives):.2f}s')
    print(f'  Max:    {np.max(half_lives):.2f}s')

> **Insight kiến trúc:**
> - **Half-life ~3–5s:** Sau 3-5 giây kể từ khi keyword xuất hiện, visual relevance
>   giảm còn 50%. Điều này phù hợp với nhịp độ thay đổi cảnh trong video.
> - **Temporal decay multiplier:** Trong `fusion_and_temporal()`, áp dụng
>   `decay_weight = exp(-lam * |t_frame - t_transcript_center| / 1000)` để boost
>   frame gần tâm transcript, giảm trọng số frame ở xa.
> - **Baseline > 0:** Ngay cả sau 10s, score vẫn > 0 — concept không biến mất hoàn toàn.
>   Điều này cho thấy nên dùng `decay_weight = max(decay, 0.2)` thay vì `exp()` thuần.
> - **Half-life thay đổi theo chủ đề:** Có thể học per-topic lamda từ dữ liệu thật
>   để điều chỉnh temporal fusion chính xác hơn.

---
## 6. Tổng Hợp: Từ EDA Đến Thiết Kế Kiến Trúc Backend

### Bảng ánh xạ EDA Insight → Fusion Strategy Decision

| Phân tích | Insight | Hành động trong `fusion_and_temporal()` |
|---|---|---|
| **Domain Gap** (Sec 1) | Một số chủ đề có cross-modal gap lớn | `per_topic_visual_weight *= 1 / domain_gap_ratio` |
| **Hubness** (Sec 2) | Frame "hub" dominate top-K, giảm precision | `confidence *= 1 / (1 + log(1 + hubness))` |
| **Query Complexity** (Sec 3) | Query phức tạp → visual score thấp, std cao | Route query L1→visual-heavy, L2→balanced, L3→text-heavy |
| **OCR-Transcript Overlap** (Sec 4) | Overlap cao → giảm text weight; thấp → tăng | `text_weight *= (1 - jaccard * 0.5)` |
| **Temporal Decay** (Sec 5) | Relevance giảm theo thời gian với half-life ~3-5s | `decay = exp(-λ * |Δt|)` với λ = 0.2–0.3 |

### Công thức Fusion đề xuất

```
final_score = w_visual * visual_score
            + w_ocr * ocr_score * overlap_penalty
            + w_transcript * transcript_score * overlap_penalty
            + w_temporal * temporal_decay_bonus

trong đó:
  w_visual, w_ocr, w_transcript = f(query_complexity, topic, domain_gap_ratio)
  overlap_penalty = 1 - jaccard(ocr, transcript) * 0.5
  temporal_decay_bonus = exp(-λ * |t_frame - t_keyword|) * hubness_penalty
```

### Ưu tiên triển khai

1. **Ngay:** Temporal decay multiplier (dễ implement, hiệu quả cao)
2. **Sớm:** Query complexity routing trong `pre_process()`
3. **Sau khi có dữ liệu thật:** Domain gap per-topic weights + Hubness penalty
4. **Nâng cao:** Học `λ_decay` và overlap penalty từ dữ liệu huấn luyện